# ImageNet Autoencoder: Simulated Data Parallelism

**Companion to [`Auto Encoder.ipynb`](Auto%20Encoder.ipynb).** That notebook built the model, the `mmap` data pipeline, and the single-GPU training baseline (10 epochs, batch 1024, ~52 s/epoch steady state). Architecture, data loading, and loss are carried over unchanged — only the *optimizer step* differs.

**Question:** This machine has one GPU. Can we still say something quantitative about what $N$-way data-parallel training would cost?

**Claim under test:** Synchronous data-parallel SGD is a statement about arithmetic, not hardware. The $N$ local gradients are all evaluated at the same $\theta_t$, so running the replicas sequentially on one card produces the *same parameter trajectory* as running them simultaneously on $N$ cards. Only wall-clock is lost — and wall-clock is the one thing we can model separately.

**Method:**
1. Clone the model $N = 5$ times.
2. Split each global batch into 5 micro-batches, one per replica.
3. Compute all 5 gradients, average them, apply one optimizer step to a master copy.
4. Broadcast the updated weights back to all 5 replicas.
5. Instrument every phase, so the measured *sequential* cost can be reassembled into a *theoretical parallel* cost.

**Hardware:**
- AMD Threadripper Pro 7965WX, 256 GB RAM
- NVIDIA RTX PRO 6000 Blackwell Workstation Edition (96 GB)
- ZFS striped 2x4TB NVMe Gen5
- Ubuntu 26.04

**Stack:** Julia / Lux.jl / CUDA.jl / Functors.jl

In [1]:
using Lux, LuxCUDA, CUDA, Optimisers, Zygote, Functors
using Random
using Statistics
using Printf

include("lib/utils.jl")
using .Utils

# Load Data

Identical to the companion notebook: `MMapReader` memory-maps the preprocessed binary shards in `build/train/` and hands back `reshape`'d views straight into the mapped buffer. No parsing, no copies, no upfront load step.

**Prerequisite:** run `Image Preprocess.ipynb` first to generate `build/train/`.

In [2]:
function load()
    base = "build/train"
	open("$base/count.ser", "r") do io
		N = deserialize(io, Int64)
		Xs = Vector{AbstractArray{Float32, 4}}(undef, N)
		μs = Vector{AbstractArray{Float32, 3}}(undef, N)
		ys = Vector{Vector{Int}}(undef, N)

		yf = open("$base/ys.ser", "r")
		Xf = MMapReader("$base/Xs.ser")
		μf = open("$base/μs.ser", "r")
        count = 0
		for i ∈ 1:N
			Xs[i] = deserialize(Xf, Array{Float32})
			μs[i] = deserialize(μf, Array{Float32})
			ys[i] = deserialize(yf, Array{Int64})
            count += length(ys[i])
		end
		close(yf); close(Xf); close(μf)
		return count, ys, Xs, μs
	end
end

@info "Loading"
@time N, ys, Xs, μs = load();
@info "||images||=$N"

[ Info: Loading


  0.051361 seconds (36.76 k allocations: 12.097 MiB, 13.05% gc time, 82.51% compilation time)


[ Info: ||images||=1281167


# Architecture

Unchanged from the companion notebook — symmetric conv/deconv, `64x64x3` → `4x4x256` → `64x64x3`. Keeping it identical is the point: any timing difference below is attributable to the optimizer step, not the network.

Note the **`BatchNorm`** layers — they turn out to matter for the equivalence argument further down.

In [3]:
encoder = Chain(
	Conv((3,3), 3 => 32, stride=2, pad=1),
	BatchNorm(32), relu,

	Conv((3,3), 32 => 64, stride=2, pad=1),
	BatchNorm(64), relu,

	Conv((3,3), 64 => 128, stride=2, pad=1),
	BatchNorm(128), relu,

	Conv((3,3), 128 => 256, stride=2, pad=1),
	BatchNorm(256), relu,
)

decoder = Chain(
	ConvTranspose((3,3), 256 => 128, stride=2, pad=1, outpad=1),
	BatchNorm(128), relu,

	ConvTranspose((3,3), 128 => 64, stride=2, pad=1, outpad=1),
	BatchNorm(64), relu,

	ConvTranspose((3,3), 64 => 32, stride=2, pad=1, outpad=1),
	BatchNorm(32), relu,

	ConvTranspose((3,3), 32 => 3, stride=2, pad=1, outpad=1),
	sigmoid
)

model = Chain(encoder, decoder)
@info "parameters = $(Lux.parameterlength(model))"

[ Info: parameters = 777987


# Averaging gradients — Data Parallel

## The mechanism

Synchronous data-parallel SGD, with $N$ workers and local batch size $B$:

$$\bar{g} = \frac{1}{N}\sum_{r=1}^{N} \nabla_\theta \mathcal{L}(\theta;\, \mathcal{B}_r), \qquad |\mathcal{B}_r| = B$$

Every worker holds the same $\theta$, sees a **different** slice $\mathcal{B}_r$ of the global batch, and computes a local gradient. An all-reduce averages them. Each worker then applies the *same* $\bar g$ with the *same* optimizer state, so all $N$ copies stay in lockstep.

## Three primitives

| step | real DDP | here |
|---|---|---|
| **scatter** | each rank reads its own shard | slice the global batch 5 ways |
| **all-reduce** | NCCL ring over NVLink/PCIe | `fmap` average over 5 gradient trees, on-device |
| **broadcast** | *not needed* — see below | `fmap(copy, ...)` into each replica |

## The broadcast is a no-op — deliberately kept

Since all replicas start identical and every replica applies the *same* averaged gradient with the *same* optimizer state, they can never diverge. Real DDP exploits exactly this: PyTorch broadcasts weights **once** at construction and thereafter all-reduces gradients only.

We keep the explicit broadcast anyway, because it makes the mechanism legible, and because timing it tells us what a parameter-server design would have cost. Its measured cost is reported separately so it can be subtracted from the theoretical parallel budget.

In [4]:
const NREP  = 5                      # simulated replicas
const MICRO = 1024                   # per-replica micro-batch (== companion notebook's batch)
const GLOBAL_BATCH = NREP * MICRO    # 5120 images per optimizer step

# Full-epoch step counts, computed from the shards rather than assumed. Both
# runs consume exactly the same images per epoch, which is what makes the
# comparison in the last section meaningful.
const STEPS_FULL_DP   = sum(size(X, 4) ÷ GLOBAL_BATCH for X in Xs)
const STEPS_FULL_BASE = sum(size(X, 4) ÷ MICRO        for X in Xs)

# Smoke-test knob. typemax(Int) = run full epochs (~8 min total). Set to e.g.
# 20 for a one-minute sanity pass first; the per-step numbers stay valid and
# the per-epoch rows are then labelled as extrapolated.
const MAX_STEPS = typemax(Int)

# --- DP primitives ---------------------------------------------------------
clone_params(p) = fmap(copy, p)

# All-reduce (mean) over N gradient trees. Lux parameter trees are nested
# NamedTuples; fmap walks them in lockstep and averages leaf-by-leaf.
avg_grads(gs) = fmap((xs...) -> sum(xs) ./ length(xs), gs...)

# CUDA kernels are async: without a sync we would be timing launch overhead,
# not execution. Every measurement below goes through this.
#
# The syncs are also load-bearing for CORRECTNESS in Run A: all NREP replicas
# stage through one `xb_gpu` buffer, so replica r's backward pass must be fully
# materialized before replica r+1 overwrites it. Don't drop them to "reduce
# overhead" without giving each replica its own buffer.
function timed(f)
    CUDA.synchronize()
    t0 = time_ns()
    r = f()
    CUDA.synchronize()
    return r, (time_ns() - t0) / 1e9
end

# --- tree helpers (shared by the verification cells and Run A) -------------
tree_leaves(x)     = [l for l in Functors.fleaves(x) if l isa AbstractArray]
tree_absmax(t)     = maximum(maximum(abs.(l)) for l in tree_leaves(t))
tree_maxdiff(a, b) = maximum(maximum(abs.(x .- y)) for (x, y) in zip(tree_leaves(a), tree_leaves(b)))

@info "N=$NREP replicas x $MICRO images = $GLOBAL_BATCH per optimizer step"
@printf("full epoch: %d DP steps vs %d baseline steps -- %d images each\n",
        STEPS_FULL_DP, STEPS_FULL_BASE, STEPS_FULL_DP * GLOBAL_BATCH)

full epoch: 250 DP steps vs 1250 baseline steps -- 1280000 images each


[ Info: N=5 replicas x 1024 images = 5120 per optimizer step


## First: is the averaged gradient actually the data-parallel gradient?

Before timing anything, verify the arithmetic on a single global batch. Two ways to consume the same 5120 images:

- **(a)** 5 replicas × 1024, gradients averaged — what this notebook does, and what 5 GPUs would do.
- **(b)** one single batch of 5120 — the "equivalent" large batch.

These are **not** identical, and the reason is `BatchNorm`. Each replica normalizes over its own 1024 samples; the single large batch normalizes over all 5120 at once. That difference is real and well known — it is precisely why `SyncBatchNorm` exists in PyTorch, and why DDP results depend on per-device batch size.

What *is* exactly equal is (a) versus real 5-GPU DDP, which is the claim that matters here.

In [5]:
let
    dev = gpu_device()
    rng = Random.default_rng(); Random.seed!(rng, 0)
    ps0, st0 = Lux.setup(rng, model)
    ps0 = ps0 |> dev; st0 = st0 |> dev

    # one global batch pulled from shard 1
    Xbig = CuArray(Xs[1][:, :, :, 1:GLOBAL_BATCH])

    function grad_of(p, xb, s)
        (l, _), back = Zygote.pullback(p) do p_
            ŷ, s2 = Lux.apply(model, xb, p_, s)
            mean((ŷ .- xb).^2), s2
        end
        return back((1f0, nothing))[1], l
    end

    # (a) data-parallel: 5 local gradients, averaged.
    # Each micro-batch is materialized contiguously rather than taken as a
    # strided @view -- cuDNN wants a dense array, and a view would silently
    # change what is being compared.
    gs = []; ls = Float32[]
    for r in 1:NREP
        lo = (r-1)*MICRO + 1
        g, l = grad_of(ps0, Xbig[:, :, :, lo:lo+MICRO-1], st0)
        push!(gs, g); push!(ls, l)
    end
    ḡ = avg_grads(gs)

    # (b) one big batch of 5120
    gbig, lbig = grad_of(ps0, Xbig, st0)

    # independent check that avg_grads is the leafwise MEAN, not the sum
    lav  = tree_leaves(ḡ)
    lrep = [tree_leaves(g) for g in gs]
    mean_err = maximum(maximum(abs.(lav[k] .- sum(lrep[r][k] for r in 1:NREP) ./ NREP))
                       for k in eachindex(lav))
    @assert mean_err == 0 "avg_grads is not the leafwise mean (err = $mean_err)"

    @printf("per-replica losses   : %s\n", string(round.(ls, digits=6)))
    @printf("mean of replicas     : %.6f\n", mean(ls))
    @printf("single 5120 batch    : %.6f\n", lbig)
    @printf("max rel |avg - big|  : %.3e   <- nonzero: BatchNorm is per-replica\n",
            tree_maxdiff(ḡ, gbig) / tree_absmax(gbig))
end

LoadError: MethodError: no method matching +(::Nothing, ::Nothing)
The function `+` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  +(::Any, ::Any, [91m::Any[39m, [91m::Any...[39m)
[0m[90m   @[39m [90mBase[39m [90m[4moperators.jl:596[24m[39m
[0m  +([91m::ChainRulesCore.ZeroTangent[39m, ::Any)
[0m[90m   @[39m [33mChainRulesCore[39m [90m~/.julia/packages/ChainRulesCore/IZ7FD/src/[39m[90m[4mtangent_arithmetic.jl:99[24m[39m
[0m  +(::Any, [91m::ChainRulesCore.NotImplemented[39m)
[0m[90m   @[39m [33mChainRulesCore[39m [90m~/.julia/packages/ChainRulesCore/IZ7FD/src/[39m[90m[4mtangent_arithmetic.jl:25[24m[39m
[0m  ...


## Second: is the *implementation* order-independent?

The gap above is the expected one, and it is the claim this notebook says is **false**. The claim it actually rests on is the other one: that visiting the replicas **sequentially** produces the same step as running them **simultaneously**.

That is testable on one GPU, without hand-waving. If there is genuinely no cross-replica data dependency within a step, then permuting the order in which the replicas are visited must leave $\theta_{t+1}$ *bit*-identical. Two things get checked here:

1. **The invariant the whole simulation depends on** — that every replica holds the same $\theta$ when its gradient is taken. If the broadcast were wrong the replicas would drift, the gradients would be evaluated at different points, and the sequential run would stop modelling anything. This must be exactly zero.
2. **Compute order vs reduction order.** Permuting which replica goes first must change nothing. Permuting the order the five gradients are *summed* in may change the last bits — floating-point addition is not associative. The third line below separates the two, and quantifies the "up to floating-point reduction order" caveat claimed at the bottom of the notebook instead of asserting it.

In [6]:
let
    dev = gpu_device()

    # five local gradients at a shared θ, replicas visited in `order`
    function dp_grads(order)
        rng = Random.default_rng(); Random.seed!(rng, 0)
        ps, st = Lux.setup(rng, model)
        ps = ps |> dev; st = st |> dev
        replicas = [clone_params(ps) for _ in 1:NREP]

        drift = maximum(tree_maxdiff(replicas[r], ps) for r in 1:NREP)

        grads = Vector{Any}(undef, NREP)
        for r in order
            lo = (r-1)*MICRO + 1
            xb = CuArray(Xs[1][:, :, :, lo:lo+MICRO-1])
            (l, _), back = Zygote.pullback(replicas[r]) do p_
                ŷ, s2 = Lux.apply(model, xb, p_, st)
                mean((ŷ .- xb).^2), s2
            end
            grads[r] = back((1f0, nothing))[1]      # stored by replica id, not visit order
        end
        return ps, grads, drift
    end

    step(ps, ḡ) = Optimisers.update(Optimisers.setup(Optimisers.Adam(1f-3), ps), ps, ḡ)[2]

    ps_f, gf, drift = dp_grads(1:NREP)
    ps_r, gr, _     = dp_grads(reverse(1:NREP))

    θ_f    = step(ps_f, avg_grads(gf))             # forward compute, forward reduce
    θ_r    = step(ps_r, avg_grads(gr))             # reversed compute, forward reduce
    θ_rr   = step(ps_r, avg_grads(reverse(gr)))    # reversed compute, reversed reduce

    @printf("replica drift at step start     : %.3e   (must be exactly 0)\n", drift)
    @printf("max |θ_fwd - θ_rev|             : %.3e   (compute order permuted)\n", tree_maxdiff(θ_f, θ_r))
    @printf("max |θ_rev - θ_rev,revreduce|   : %.3e   (reduction order permuted)\n", tree_maxdiff(θ_r, θ_rr))
    @printf("θ scale (max |θ|)               : %.3e\n", tree_absmax(θ_f))
    println()
    println(tree_maxdiff(θ_f, θ_r) == 0 ?
        "Compute order is not load-bearing: the sequential loop IS the parallel step." :
        "Compute order changed θ -- the simulation has a hidden cross-replica dependency.")
end

LoadError: MethodError: no method matching +(::Nothing, ::Nothing)
The function `+` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  +(::Any, ::Any, [91m::Any[39m, [91m::Any...[39m)
[0m[90m   @[39m [90mBase[39m [90m[4moperators.jl:596[24m[39m
[0m  +([91m::ChainRulesCore.ZeroTangent[39m, ::Any)
[0m[90m   @[39m [33mChainRulesCore[39m [90m~/.julia/packages/ChainRulesCore/IZ7FD/src/[39m[90m[4mtangent_arithmetic.jl:99[24m[39m
[0m  +(::Any, [91m::ChainRulesCore.NotImplemented[39m)
[0m[90m   @[39m [33mChainRulesCore[39m [90m~/.julia/packages/ChainRulesCore/IZ7FD/src/[39m[90m[4mtangent_arithmetic.jl:25[24m[39m
[0m  ...


# The timing model

One GPU runs the 5 replicas back to back, so measured wall-clock per step is

$$t_{\text{seq}} = \sum_{r=1}^{N}\bigl(t^{\text{copy}}_r + t^{\text{grad}}_r\bigr) \;+\; t_{\text{reduce}} + t_{\text{update}} + t_{\text{bcast}}$$

On $N$ real GPUs those per-replica terms happen **concurrently**, the on-device average is replaced by an inter-GPU all-reduce, and the broadcast disappears:

$$t_{\text{par}} = \max_{r}\bigl(t^{\text{copy}}_r + t^{\text{grad}}_r\bigr) \;+\; t_{\text{allreduce}} + t_{\text{update}}$$

The only term we cannot measure on one card is $t_{\text{allreduce}}$, so we model it. A ring all-reduce completes in $2(N-1)$ hops and moves $2\,P\,s\,(N-1)/N$ bytes per GPU, for $P$ parameters of $s$ bytes each:

$$t_{\text{allreduce}} = \underbrace{2(N-1)\,\alpha}_{\text{per-hop latency}} \;+\; \underbrace{\frac{2(N-1)}{N}\cdot\frac{Ps}{\beta}}_{\text{bandwidth}}$$

This model has ~0.78 M parameters, so $Ps \approx 3.1$ MB and each GPU ships ~5 MB per step. At $\beta = 64$ GB/s that is ~78 µs of transfer — but the payload is small enough that the latency term is the *same order*: 8 hops at $\alpha \approx 5$ µs is ~40 µs. A bandwidth-only model would understate the all-reduce by about half, so both terms are exposed below (`LINK_GBPS`, `LINK_LATENCY_US`). Against a forward/backward over 1024 images (~40 ms) the whole thing is a rounding error either way — which is the useful conclusion, and the reason the result is insensitive to the choice of NVLink vs PCIe.

**This is a ceiling, not a benchmark.** It assumes perfectly balanced replicas, models the all-reduce rather than measuring it, and ignores straggler effects, kernel-launch contention, and host-bandwidth contention from a real second GPU.

In [7]:
const PARAM_BYTES = Lux.parameterlength(model) * 4      # Float32

# Interconnect model. 64 GB/s is PCIe 5.0 x16 *raw* unidirectional -- achievable
# all-reduce bandwidth through a root complex is well below it, NVLink well above.
# LINK_LATENCY_US is the per-hop cost (kernel launch + collective handshake); it
# matters here only because the payload is so small.
const LINK_GBPS       = 64.0
const LINK_LATENCY_US = 5.0

# Ring all-reduce: 2(N-1) hops, 2(N-1)/N * payload bytes moved per GPU.
function allreduce_seconds(nrep)
    nrep <= 1 && return 0.0
    lat = 2 * (nrep - 1) * LINK_LATENCY_US * 1e-6
    bw  = 2 * PARAM_BYTES * (nrep - 1) / nrep / (LINK_GBPS * 1e9)
    return lat + bw
end

allreduce_bytes(nrep) = 2 * PARAM_BYTES * (nrep - 1) / nrep

@printf("parameters        : %d (%.2f MB as Float32)\n", Lux.parameterlength(model), PARAM_BYTES/1e6)
@printf("moved per GPU     : %.2f MB per step\n", allreduce_bytes(NREP)/1e6)
@printf("ring all-reduce   : %.3f ms  = %.0f us latency (%d hops) + %.0f us transfer at %.0f GB/s\n",
        allreduce_seconds(NREP)*1e3,
        2*(NREP-1)*LINK_LATENCY_US, 2*(NREP-1),
        allreduce_bytes(NREP)/(LINK_GBPS*1e9)*1e6, LINK_GBPS)

parameters        : 777987 (3.11 MB as Float32)
moved per GPU     : 4.98 MB per step
ring all-reduce   : 0.118 ms  = 40 us latency (8 hops) + 78 us transfer at 64 GB/s


# Run A — simulated data parallel

$N = 5$ replicas, global batch 5120, instrumented per phase. Two epochs is enough for a stable per-step cost; the companion notebook's epoch 1 was ~2x slower than steady state purely from JIT compilation, so **epoch 1 is excluded from the timing summary**.

Set `MAX_STEPS` in the primitives cell to a small number (e.g. 20) for a quick sanity pass before committing to the full ~8 minutes; the per-step measurements stay valid and the per-epoch rows get labelled as extrapolated.

In [8]:
dp_stats = Dict{Symbol, Any}()

@info "Run A: simulated data parallel (N=$NREP)"
let
    dev = gpu_device()
    rng = Random.default_rng(); Random.seed!(rng, 0)
    ps, st = Lux.setup(rng, model)
    ps = ps |> dev; st = st |> dev

    opt_state = Optimisers.setup(Optimisers.Adam(1f-3), ps)
    replicas  = [clone_params(ps) for _ in 1:NREP]
    xb_gpu    = CUDA.zeros(Float32, 64, 64, 3, MICRO)   # shared staging buffer; see `timed`

    nepochs = 2
    acc = (copy=Float64[], grad=Float64[], gradmax=Float64[],
           reduce=Float64[], update=Float64[], bcast=Float64[], step=Float64[])

    for epoch in 1:nepochs
        epoch_loss = 0.0f0; nsteps = 0
        epoch_start = time()
        warm = epoch > 1                       # epoch 1 is JIT warm-up
        drift_checked = false

        for k in randperm(length(Xs))
            nsteps >= MAX_STEPS && break
            Xk = Xs[k]; Nk = size(Xk, 4)
            starts = collect(1:GLOBAL_BATCH:Nk)
            starts = starts[randperm(length(starts))]

            for i in starts
                i + GLOBAL_BATCH - 1 > Nk && continue
                nsteps >= MAX_STEPS && break
                step_t0 = time_ns()

                # The invariant the whole simulation rests on: every replica must
                # hold the master θ when its gradient is taken, or the N gradients
                # are not being evaluated at a common θ_t and nothing below models
                # anything. Checked once per epoch -- cheap, but not free.
                if !drift_checked
                    drift = maximum(tree_maxdiff(replicas[r], ps) for r in 1:NREP)
                    @assert drift == 0 "replicas drifted from master by $drift -- broadcast is broken"
                    drift_checked = true
                end

                grads  = Vector{Any}(undef, NREP)
                losses = Vector{Float32}(undef, NREP)
                tcopy  = zeros(Float64, NREP)
                tgrad  = zeros(Float64, NREP)
                st_next = st

                # --- scatter + local gradients (sequential here, concurrent on N GPUs)
                for r in 1:NREP
                    lo = i + (r-1)*MICRO
                    _, tc = timed() do
                        copyto!(xb_gpu, Xk[:, :, :, lo:lo+MICRO-1])
                    end
                    (g, l, s_new), tg = timed() do
                        (l, s), back = Zygote.pullback(replicas[r]) do p_
                            ŷ, s2 = Lux.apply(model, xb_gpu, p_, st)
                            mean((ŷ .- xb_gpu).^2), s2
                        end
                        back((1f0, nothing))[1], l, s
                    end
                    grads[r] = g; losses[r] = l
                    tcopy[r] = tc; tgrad[r] = tg
                    r == 1 && (st_next = s_new)   # BN buffers follow replica 1 (cf. DDP broadcast_buffers)
                end

                # --- all-reduce
                ḡ, tred = timed(() -> avg_grads(grads))

                # --- master update
                (opt_state, ps), tupd = timed() do
                    Optimisers.update(opt_state, ps, ḡ)
                end

                # --- broadcast back into every replica (a no-op in real DDP)
                _, tbc = timed() do
                    for r in 1:NREP; replicas[r] = clone_params(ps); end
                end

                st = st_next
                epoch_loss += mean(losses); nsteps += 1

                if warm
                    push!(acc.copy,    sum(tcopy))
                    push!(acc.grad,    sum(tgrad))
                    push!(acc.gradmax, maximum(tcopy .+ tgrad))
                    push!(acc.reduce,  tred)
                    push!(acc.update,  tupd)
                    push!(acc.bcast,   tbc)
                    push!(acc.step,    (time_ns() - step_t0) / 1e9)
                end
            end
        end

        elapsed = time() - epoch_start
        @printf("Epoch %d/%d | avg loss %.5f | %.1fs | %d steps | %d imgs/s%s\n",
                epoch, nepochs, epoch_loss/nsteps, elapsed, nsteps,
                round(Int, nsteps*GLOBAL_BATCH/elapsed), warm ? "" : "  (warm-up, excluded)")
        if warm
            dp_stats[:epoch_seconds]   = elapsed
            dp_stats[:steps_per_epoch] = nsteps
            dp_stats[:final_loss]      = epoch_loss/nsteps
        end
    end

    dp_stats[:t] = acc
end;

[ Info: Run A: simulated data parallel (N=5)


LoadError: MethodError: no method matching +(::Nothing, ::Nothing)
The function `+` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  +(::Any, ::Any, [91m::Any[39m, [91m::Any...[39m)
[0m[90m   @[39m [90mBase[39m [90m[4moperators.jl:596[24m[39m
[0m  +([91m::ChainRulesCore.ZeroTangent[39m, ::Any)
[0m[90m   @[39m [33mChainRulesCore[39m [90m~/.julia/packages/ChainRulesCore/IZ7FD/src/[39m[90m[4mtangent_arithmetic.jl:99[24m[39m
[0m  +(::Any, [91m::ChainRulesCore.NotImplemented[39m)
[0m[90m   @[39m [33mChainRulesCore[39m [90m~/.julia/packages/ChainRulesCore/IZ7FD/src/[39m[90m[4mtangent_arithmetic.jl:25[24m[39m
[0m  ...


## Reassembling the measurements into a theoretical parallel time

Everything above was measured *sequentially*. Now re-add the per-phase costs under the parallel model: the five replica gradients collapse to the slowest one, the on-device average is replaced by the modelled ring all-reduce, and the broadcast drops out.

In [9]:
let
    @assert haskey(dp_stats, :t) "run Run A first"
    t = dp_stats[:t]
    @assert !isempty(t.step) "no steady-state steps recorded -- raise MAX_STEPS, or Run A stopped inside epoch 1"
    nsteps = dp_stats[:steps_per_epoch]

    seq_step = mean(t.step)
    par_step = mean(t.gradmax) + allreduce_seconds(NREP) + mean(t.update)

    @printf("--- measured, per optimizer step (steady state, n=%d) ---\n", length(t.step))
    @printf("  H->D copy   x%d  : %7.2f ms   (mean single: %6.2f ms)\n", NREP, mean(t.copy)*1e3, mean(t.copy)/NREP*1e3)
    @printf("  fwd+bwd     x%d  : %7.2f ms   (mean single: %6.2f ms)\n", NREP, mean(t.grad)*1e3, mean(t.grad)/NREP*1e3)
    @printf("  all-reduce (dev) : %7.2f ms\n", mean(t.reduce)*1e3)
    @printf("  master update    : %7.2f ms\n", mean(t.update)*1e3)
    @printf("  broadcast   x%d  : %7.2f ms   (free in real DDP)\n", NREP, mean(t.bcast)*1e3)
    @printf("  total step       : %7.2f ms\n", seq_step*1e3)

    @printf("\n--- reassembled as %d GPUs ---\n", NREP)
    @printf("  slowest replica  : %7.2f ms   (max over r of copy + grad)\n", mean(t.gradmax)*1e3)
    @printf("  ring all-reduce  : %7.2f ms   (modelled: %.0f GB/s, %.0f us/hop)\n",
            allreduce_seconds(NREP)*1e3, LINK_GBPS, LINK_LATENCY_US)
    @printf("  master update    : %7.2f ms\n", mean(t.update)*1e3)
    @printf("  total step       : %7.2f ms\n", par_step*1e3)

    @printf("\n  sequential sim -> modelled parallel : %.2fx\n", seq_step/par_step)
    println("  (this ratio is NOT the headline: seq_step carries the broadcast and")
    println("   the measurement syncs, neither of which a real DDP step pays. The")
    println("   comparison that means something is against Run B, below.)")

    dp_stats[:seq_step] = seq_step
    dp_stats[:par_step] = par_step
    dp_stats[:theoretical_epoch] = STEPS_FULL_DP * par_step     # always a FULL epoch
end

LoadError: AssertionError: run Run A first

# Run B — the baseline, re-measured here

The companion notebook reported ~44 s/epoch, but that was a different process with a different warm-up history. To compare honestly we re-run the original single-model loop **in this session**, same two epochs, same data order policy, batch 1024.

In [10]:
base_stats = Dict{Symbol, Any}()

@info "Run B: single-model baseline (batch $MICRO)"
let
    dev = gpu_device()
    rng = Random.default_rng(); Random.seed!(rng, 0)
    ps, st = Lux.setup(rng, model)
    ps = ps |> dev; st = st |> dev

    opt_state = Optimisers.setup(Optimisers.Adam(1f-3), ps)
    xb_gpu = CUDA.zeros(Float32, 64, 64, 3, MICRO)

    nepochs = 2
    # cover the same images as a capped Run A: NREP times the steps
    max_base = MAX_STEPS > typemax(Int) ÷ NREP ? typemax(Int) : MAX_STEPS * NREP

    for epoch in 1:nepochs
        epoch_loss = 0.0f0; nbatches = 0
        epoch_start = time()
        warm = epoch > 1

        for k in randperm(length(Xs))
            nbatches >= max_base && break
            Xk = Xs[k]; Nk = size(Xk, 4)
            starts = collect(1:MICRO:Nk)
            starts = starts[randperm(length(starts))]

            for i in starts
                j = min(i + MICRO - 1, Nk)
                (j - i + 1) < MICRO && continue
                nbatches >= max_base && break
                copyto!(xb_gpu, Xk[:, :, :, i:j])

                (loss, st_new), back = Zygote.pullback(ps) do p_
                    ŷ, s = Lux.apply(model, xb_gpu, p_, st)
                    mean((ŷ .- xb_gpu).^2), s
                end
                ∇l = back((1f0, nothing))[1]
                opt_state, ps = Optimisers.update(opt_state, ps, ∇l)
                st = st_new

                epoch_loss += loss; nbatches += 1
            end
        end

        elapsed = time() - epoch_start
        @printf("Epoch %d/%d | avg loss %.5f | %.1fs | %d batches | %d imgs/s%s\n",
                epoch, nepochs, epoch_loss/nbatches, elapsed, nbatches,
                round(Int, nbatches*MICRO/elapsed), warm ? "" : "  (warm-up, excluded)")
        if warm
            base_stats[:epoch_seconds]   = elapsed
            base_stats[:steps_per_epoch] = nbatches
            base_stats[:final_loss]      = epoch_loss/nbatches
        end
    end
end;

[ Info: Run B: single-model baseline (batch 1024)


Epoch 1/2 | avg loss 0.00735 | 55.8s | 1250 batches | 22943 imgs/s  (warm-up, excluded)
Epoch 2/2 | avg loss 0.00336 | 50.6s | 1250 batches | 25298 imgs/s


# Comparison

In [11]:
let
    b, d = base_stats, dp_stats
    @assert haskey(b, :epoch_seconds) "run Run B first"
    @assert haskey(d, :par_step)      "run Run A and the reassembly cell first"

    b_step = b[:epoch_seconds] / b[:steps_per_epoch]      # measured, per optimizer step
    d_step = d[:epoch_seconds] / d[:steps_per_epoch]
    partial = b[:steps_per_epoch] < STEPS_FULL_BASE || d[:steps_per_epoch] < STEPS_FULL_DP

    b_epoch = b_step * STEPS_FULL_BASE                    # full-epoch equivalents
    d_epoch = d_step * STEPS_FULL_DP
    d_theor = d[:theoretical_epoch]

    @printf("%-34s %14s %14s\n", "", "baseline", "data parallel")
    @printf("%-34s %14s %14s\n", "-"^34, "-"^14, "-"^14)
    @printf("%-34s %14d %14d\n", "images per optimizer step", MICRO, GLOBAL_BATCH)
    @printf("%-34s %14d %14d\n", "optimizer steps per full epoch", STEPS_FULL_BASE, STEPS_FULL_DP)
    @printf("%-34s %14d %14d\n", "images per full epoch", STEPS_FULL_BASE*MICRO, STEPS_FULL_DP*GLOBAL_BATCH)
    @printf("%-34s %14d %14d\n", "steps actually measured", b[:steps_per_epoch], d[:steps_per_epoch])
    @printf("%-34s %14.2f %14.2f\n", "measured ms per step (1 GPU)", b_step*1e3, d_step*1e3)
    @printf("%-34s %14.1f %14.1f\n", "epoch (s, 1 GPU)", b_epoch, d_epoch)
    @printf("%-34s %14s %14.1f\n", "epoch (s, modelled multi-GPU)", "n/a", d_theor)
    @printf("%-34s %14.5f %14.5f\n", "loss after 2 epochs", b[:final_loss], d[:final_loss])
    println()
    @printf("sequential DP vs baseline      : %+.1f%%  (identical fwd/bwd work, plus reduce + broadcast)\n",
            100*(d_epoch/b_epoch - 1))
    @printf("modelled %d-GPU vs baseline     : %.2fx faster per epoch  (a ceiling, not a benchmark)\n",
            NREP, b_epoch/d_theor)
    if partial
        println()
        println("NOTE: MAX_STEPS capped the run. Per-epoch rows are measured per-step cost")
        println("      scaled to the full step count, not wall-clock over a whole epoch.")
    end
end

LoadError: AssertionError: run Run A and the reassembly cell first

# Is this mathematically the same as synchronous SGD?

**Yes — and that is not hand-waving, it is the definition.**

Synchronous data-parallel SGD *is* the update

$$\theta_{t+1} = \theta_t - \eta\,\frac{1}{N}\sum_{r=1}^{N}\nabla_\theta\mathcal{L}(\theta_t;\,\mathcal{B}_r)$$

Nothing in that expression mentions a device. The $N$ gradients are evaluated at the **same** $\theta_t$, so they are mutually independent — there is no data dependency between replica $r$ and replica $r'$ within a step. Independent work may be executed in any order, including sequentially, without changing the result. A 5-GPU run and this loop produce the same $\theta_{t+1}$, up to floating-point reduction order.

So the honest framing is: **the simulation is exact for the mathematics and approximate only for the timing.** We are not hand-waving that the replicas ran in parallel; we are observing that parallelism was never load-bearing for the arithmetic in the first place.

The order-invariance cell near the top is the empirical half of that argument: permuting which replica is visited first leaves $\theta_{t+1}$ bit-identical, and the only thing the *reduction* order moves is the last few bits of the average.

### Three things that are *not* the same, and are worth stating

1. **Not the same as one batch of $NB$.** `BatchNorm` normalizes per replica, over $B$ samples, not over $NB$. The cell near the top measures this gap directly. `SyncBatchNorm` exists precisely to close it, at the cost of an extra all-reduce per BN layer.

2. **Not the same as $N$ sequential SGD steps of batch $B$.** This is one step on an averaged gradient, not $N$ steps. Per epoch it takes $N$ times **fewer** parameter updates. Gradient variance falls like $1/N$, but you spend fewer updates — which is why large-batch training usually needs the linear scaling rule ($\eta \to N\eta$) plus warm-up (Goyal et al., 2017) to match small-batch convergence. We deliberately did **not** rescale $\eta$ here, so the loss row in the comparison above is expected to favour the baseline; this experiment is about time, not convergence.

3. **Not a measurement of multi-GPU performance.** $t_{\text{par}}$ assumes perfectly balanced replicas, models the all-reduce rather than measuring it, and ignores stragglers, launch contention, and host-bandwidth contention. It is a ceiling.

4. **BatchNorm *buffers* follow replica 1.** Running means and variances are state, not parameters, so they are never all-reduced; Run A carries replica 1's forward and discards the other four. That is also what PyTorch DDP does by default — `broadcast_buffers=True` pushes rank 0's buffers out at every forward — so the choice tracks real DDP rather than departing from it. It is still a choice, and `SyncBatchNorm` is what removes the need to make it.

# Takeaway

- The arithmetic of synchronous data parallelism carries over to one GPU **exactly**. Only wall-clock needs modelling, and the model has one free parameter (`LINK_GBPS`) that this network is too small to be sensitive to — at ~0.78 M parameters the ring all-reduce is a rounding error next to a forward/backward over 1024 images.

- Running the replicas sequentially costs *more* than the baseline, not less: identical total forward/backward work, plus the reduce and the broadcast. That is the expected sign, and it is why the theoretical reassembly — not the measured wall-clock — is the output of this notebook.

- The explicit weight broadcast is provably redundant. Real DDP broadcasts once at init and all-reduces gradients thereafter; we timed it only to show what it would have cost.

**Carried over from the companion notebook:** that run reported `606 GiB` allocated and `11.88% gc time`. The cause is `Xk[:, :, :, i:j]`, which materializes a fresh 50 MB host array per batch rather than viewing the mmap — and, being pageable, also forces every `copyto!` to a synchronous transfer. That is the likely reason the async-stream double-buffering experiment did not help. Both notebooks keep the original formulation so the comparison stays apples-to-apples; fixing it is the obvious next experiment.